# 🧠 Introducción a RAG — Retrieval Augmented Generation
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

En este notebook vamos a recorrer el camino que va desde entender **por qué los LLMs tienen limitaciones estructurales** hasta comprender cómo RAG las resuelve.

Al final de este notebook vas a poder responder:
- ¿Por qué un LLM puede "mentir" con total confianza?
- ¿Cómo genera texto un modelo de lenguaje por dentro?
- ¿Qué es RAG y cómo resuelve esas limitaciones?
- ¿Qué alternativas y evoluciones existen hoy?

## ⚙️ Instalación de dependencias

Ejecutá esta celda primero. Instalamos todo lo que vamos a necesitar en el notebook.

In [1]:
!pip install openai tiktoken --quiet
print('✅ Dependencias instaladas')

✅ Dependencias instaladas


In [2]:
import os
from openai import OpenAI

# 🔑 Ingresá tu API key de OpenAI
# En Colab podés usar: from google.colab import userdata
# y guardar la key en Secrets con el nombre OPENAI_API_KEY

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.getenv('OPENAI_API_KEY', 'TU_API_KEY_ACÁ')

client = OpenAI(api_key=api_key)
print('✅ Cliente OpenAI inicializado')

✅ Cliente OpenAI inicializado


---
# 📌 Parte 1 — ¿Por qué los LLMs mienten con tanta confianza?

Un LLM como GPT-4 fue entrenado con texto de internet hasta cierta fecha. Eso implica tres limitaciones estructurales:

| Problema | Descripción | Ejemplo |
|---|---|---|
| **Knowledge cutoff** | No sabe nada posterior a su fecha de entrenamiento | "¿Quién ganó las elecciones de 2025?" |
| **Alucinación** | Genera texto plausible aunque sea falso | Inventa citas, leyes, personas |
| **Conocimiento privado** | No tiene acceso a documentos internos de tu empresa | Manuales, políticas, stock |

Veamos el problema en acción 👇

In [3]:
# 🔴 DEMO: El LLM sin contexto — observá la respuesta

def preguntar_sin_contexto(pregunta):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": pregunta}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Pregunta sobre información que el modelo no puede tener
pregunta = """¿Cuál es la política de devoluciones de la empresa TechStore Argentina
para productos electrónicos comprados online?"""

print("❓ Pregunta:", pregunta)
print("\n🤖 Respuesta del LLM sin contexto:")
print("-" * 60)
respuesta = preguntar_sin_contexto(pregunta)
print(respuesta)
print("-" * 60)
print("\n⚠️  El modelo respondió con confianza, pero ¿de dónde sacó esa información?")
print("    Simplemente generó texto plausible. Eso es alucinación.")

❓ Pregunta: ¿Cuál es la política de devoluciones de la empresa TechStore Argentina 
para productos electrónicos comprados online?

🤖 Respuesta del LLM sin contexto:
------------------------------------------------------------
No tengo acceso a información actualizada en tiempo real, así que te recomendaría que consultes directamente el sitio web de TechStore Argentina o contactes a su servicio al cliente para obtener la información más precisa y actual sobre su política de devoluciones para productos electrónicos comprados online. Generalmente, las políticas de devoluciones pueden incluir aspectos como el plazo para realizar una devolución, el estado en que deben estar los productos, y si se ofrecen reembolsos o cambios.
------------------------------------------------------------

⚠️  El modelo respondió con confianza, pero ¿de dónde sacó esa información?
    Simplemente generó texto plausible. Eso es alucinación.


### 🔍 ¿Qué observás en la respuesta anterior?

El modelo respondió con total confianza sobre una empresa ficticia. No dijo "no sé", inventó una política que suena razonable.

Esto no es un bug, es una consecuencia directa de **cómo funciona el modelo por dentro**. Para entenderlo hay que abrir la caja negra.

---
# 📌 Parte 2 — Cómo funciona un LLM por dentro

## 2.1 — La idea central: predicción del siguiente token

Un LLM no "piensa" ni "entiende" en el sentido humano. Su mecanismo central es:

```
Dado el texto que vino antes → ¿cuál es el token más probable que sigue?
```

Eso es todo. La "inteligencia" emerge de hacer eso extremadamente bien, con billones de parámetros y enormes cantidades de texto.

## 2.2 — El loop autoregresivo

```
  Entrada: ["El", "cielo", "es"]
       ↓
  Transformer (todas las capas de atención)
       ↓
  Vector de logits (uno por cada token del vocabulario ~50.000 tokens)
       ↓
  Softmax → distribución de probabilidad
       ↓
  Sampleo → "azul"  (token elegido)
       ↓
  Nueva entrada: ["El", "cielo", "es", "azul"]
       ↓
  ... repite hasta token de fin de secuencia
```

**Punto clave:** el modelo se alimenta de su propio output. Cada token generado se convierte en parte de la entrada para el siguiente.

## 2.3 — Softmax y Temperature

La función Softmax convierte los logits crudos en probabilidades que suman 1.

El parámetro **temperature** escala los logits *antes* del Softmax:

| Temperature | Efecto | Uso típico |
|---|---|---|
| `0` | Siempre el token más probable (greedy) | Respuestas deterministas |
| `0.2 - 0.5` | Distribución afilada, poco azar | Respuestas precisas |
| `0.7 - 1.0` | Distribución balanceada | Uso general |
| `1.5+` | Distribución muy plana, mucho azar | Creatividad, brainstorming |

Veamos el efecto en vivo 👇

In [4]:
# 🎛️ DEMO: El efecto de la temperature

def generar_con_temperature(prompt, temperature, n=1):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=80
    )
    return response.choices[0].message.content

prompt = "Continuá esta oración de forma creativa: 'El vendedor miró el inventario y...' "

temperatures = [0.0, 0.7, 1.5]

print(f"📝 Prompt: {prompt}\n")
print("=" * 60)

for temp in temperatures:
    print(f"\n🌡️  Temperature = {temp}")
    print("-" * 40)
    # Generamos 2 veces para mostrar la variabilidad
    for i in range(2):
        resultado = generar_con_temperature(prompt, temp)
        print(f"  Intento {i+1}: {resultado}")

📝 Prompt: Continuá esta oración de forma creativa: 'El vendedor miró el inventario y...' 


🌡️  Temperature = 0.0
----------------------------------------
  Intento 1: El vendedor miró el inventario y, con una sonrisa traviesa, se dio cuenta de que entre los productos comunes había un objeto inusual: una antigua brújula que parecía susurrar secretos del pasado. Intrigado, decidió que no podía dejarla en el estante, así que la tomó y, al hacerlo, sintió una corriente de energía recorrer su cuerpo. Sin pens
  Intento 2: El vendedor miró el inventario y, con una sonrisa traviesa, se dio cuenta de que entre los objetos polvorientos y olvidados había una antigua caja de música. Al abrirla, una melodía suave y nostálgica llenó el aire, y de repente, los recuerdos de su infancia lo invadieron. Decidió que no podía dejarla pasar; esa caja tenía

🌡️  Temperature = 0.7
----------------------------------------
  Intento 1: El vendedor miró el inventario y frunció el ceño al ver que solo quedaban 

### 🔍 ¿Qué observás?

- Con **temperature 0**: las dos respuestas son idénticas o casi idénticas. El modelo siempre elige el token más probable.
- Con **temperature 0.7**: hay variación pero las respuestas siguen siendo coherentes.
- Con **temperature 1.5**: las respuestas pueden ser muy distintas entre sí, más creativas o incluso incoherentes.

> 💡 **Esto explica algo importante sobre RAG:** el mismo sistema RAG puede dar respuestas distintas a la misma pregunta. No es un bug, es la naturaleza probabilística del generador.

In [5]:
# 🔬 EXPERIMENTO LIBRE — Temperatura y tokens
# Modificá el prompt y las temperatures para explorar el comportamiento
import tiktoken

def analizar_respuesta(prompt, temperature):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=150
    )
    texto = response.choices[0].message.content
    tokens_entrada = response.usage.prompt_tokens
    tokens_salida = response.usage.completion_tokens
    costo_aprox = (tokens_entrada * 0.00000015) + (tokens_salida * 0.0000006)

    print(f"🌡️  Temperature: {temperature}")
    print(f"📥 Tokens de entrada: {tokens_entrada}")
    print(f"📤 Tokens de salida: {tokens_salida}")
    print(f"💰 Costo aproximado: ${costo_aprox:.6f} USD")
    print(f"\n💬 Respuesta:\n{texto}")
    print("=" * 60)

mi_prompt = "Explicá en 3 oraciones qué es el aprendizaje automático."
analizar_respuesta(mi_prompt, temperature=0.0)
analizar_respuesta(mi_prompt, temperature=1.0)

🌡️  Temperature: 0.0
📥 Tokens de entrada: 21
📤 Tokens de salida: 87
💰 Costo aproximado: $0.000055 USD

💬 Respuesta:
El aprendizaje automático es una rama de la inteligencia artificial que permite a las computadoras aprender y mejorar su rendimiento en tareas específicas a partir de datos, sin ser programadas explícitamente para cada tarea. Utiliza algoritmos y modelos estadísticos para identificar patrones y hacer predicciones basadas en la información proporcionada. Este enfoque se aplica en diversas áreas, como el reconocimiento de voz, la visión por computadora y la recomendación de productos.
🌡️  Temperature: 1.0
📥 Tokens de entrada: 21
📤 Tokens de salida: 105
💰 Costo aproximado: $0.000066 USD

💬 Respuesta:
El aprendizaje automático es una rama de la inteligencia artificial que se centra en el desarrollo de algoritmos y modelos que permiten a las computadoras aprender a partir de datos sin ser programadas explícitamente. A través de procesos como la identificación de patrones y la 

---
# 📌 Parte 3 — El pipeline RAG completo

Ahora que entendemos las limitaciones del LLM y cómo funciona por dentro, RAG tiene sentido como solución.

**La idea central de RAG:**

> En vez de esperar que el modelo "sepa" la respuesta, le *damos* la información relevante en el prompt.

## El pipeline en dos etapas

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ETAPA 1 — INDEXACIÓN (se hace una vez, offline)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Documentos → Chunking → Embeddings → Vector Store
  
  📄📄📄         ✂️✂️✂️       🔢🔢🔢        🗄️
  textos       fragmentos   vectores    índice
  crudos       de texto     numéricos   buscable

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ETAPA 2 — RECUPERACIÓN Y GENERACIÓN (en tiempo real)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Pregunta → Embedding → Similitud → Top-K chunks
                                          ↓
                              Prompt aumentado
                              = instrucción + contexto + pregunta
                                          ↓
                                        LLM
                                          ↓
                                      Respuesta
```

## Los componentes y sus decisiones de diseño

| Componente | Qué hace | Decisiones que afectan la calidad |
|---|---|---|
| **Chunking** | Divide documentos en fragmentos | Tamaño, overlap, estrategia |
| **Embeddings** | Convierte texto en vectores numéricos | Modelo elegido, dimensiones |
| **Vector Store** | Guarda y busca vectores | Top-K, métrica de similitud |
| **Prompt aumentado** | Arma la consulta final al LLM | Instrucciones, formato del contexto |
| **LLM generador** | Produce la respuesta final | Modelo, temperature, max_tokens |

> 💡 **Cada una de estas decisiones tiene un Colab dedicado.** En este notebook solo mostramos el pipeline completo funcionando de punta a punta.

In [6]:
# 📦 Instalamos las librerías del pipeline RAG
!pip install chromadb sentence-transformers --quiet
print('✅ ChromaDB y sentence-transformers instalados')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [7]:
# 📄 CORPUS DE EJEMPLO — Política de empresa ficticia
# En los Colabs siguientes vas a poder reemplazar esto con tus propios documentos

documentos = [
    """POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos
    desde la fecha de compra, siempre que presenten el ticket original y el producto
    esté en su embalaje original sin señales de uso.""",

    """GARANTÍA DE PRODUCTOS — TechStore Argentina
    Todos los productos tienen garantía legal de 6 meses por defectos de fabricación.
    Los televisores y notebooks tienen garantía extendida de 12 meses.
    La garantía no cubre daños por mal uso o accidentes.""",

    """POLÍTICA DE ENVÍOS — TechStore Argentina
    Los envíos al interior del país tienen un plazo de 3 a 7 días hábiles.
    El envío es gratuito para compras superiores a $50.000.
    Para CABA y GBA el plazo es de 24 a 48 horas hábiles.""",

    """MEDIOS DE PAGO — TechStore Argentina
    Aceptamos tarjetas de crédito Visa, Mastercard y American Express.
    Las compras en 12 cuotas sin interés aplican solo para productos seleccionados.
    También aceptamos transferencia bancaria con un 10% de descuento adicional.""",

    """ATENCIÓN AL CLIENTE — TechStore Argentina
    Nuestro centro de atención opera de lunes a viernes de 9 a 18 hs.
    Podés contactarnos por WhatsApp al +54 11 4000-0000 o por email a soporte@techstore.com.ar.
    Los reclamos se resuelven en un plazo máximo de 72 horas hábiles."""
]

print(f"📄 Corpus cargado: {len(documentos)} documentos")
for i, doc in enumerate(documentos):
    print(f"  Documento {i+1}: {doc.split(chr(10))[1].strip()[:50]}...")

📄 Corpus cargado: 5 documentos
  Documento 1: Los clientes pueden devolver productos electrónico...
  Documento 2: Todos los productos tienen garantía legal de 6 mes...
  Documento 3: Los envíos al interior del país tienen un plazo de...
  Documento 4: Aceptamos tarjetas de crédito Visa, Mastercard y A...
  Documento 5: Nuestro centro de atención opera de lunes a vierne...


In [8]:
# ✂️ ETAPA 1A — CHUNKING
# Por ahora usamos los documentos completos como chunks
# En el Colab 1 vamos a explorar distintas estrategias

chunks = documentos  # cada documento es un chunk
print(f"✅ Chunking simple: {len(chunks)} chunks generados")
print(f"   Tamaño promedio: {sum(len(c) for c in chunks) // len(chunks)} caracteres por chunk")

✅ Chunking simple: 5 chunks generados
   Tamaño promedio: 263 caracteres por chunk


In [9]:
# 🔢 ETAPA 1B — EMBEDDINGS + VECTOR STORE
# Usamos ChromaDB con embeddings de OpenAI

import chromadb
from chromadb.utils import embedding_functions

# Función de embedding usando OpenAI
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=api_key,
    model_name="text-embedding-3-small"
)

# Creamos el vector store en memoria
chroma_client = chromadb.Client()

# Eliminamos la colección si ya existe (para poder re-ejecutar la celda)
try:
    chroma_client.delete_collection("techstore_politicas")
except:
    pass

coleccion = chroma_client.create_collection(
    name="techstore_politicas",
    embedding_function=openai_ef
)

# Indexamos los chunks
coleccion.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Vector store creado con {coleccion.count()} documentos indexados")
print(f"   Modelo de embeddings: text-embedding-3-small")
print(f"   Métrica de similitud: coseno (por defecto en ChromaDB)")

✅ Vector store creado con 5 documentos indexados
   Modelo de embeddings: text-embedding-3-small
   Métrica de similitud: coseno (por defecto en ChromaDB)


In [10]:
# 🔍 ETAPA 2A — RECUPERACIÓN
# Dada una pregunta, buscamos los chunks más relevantes

def recuperar_contexto(pregunta, top_k=2):
    resultados = coleccion.query(
        query_texts=[pregunta],
        n_results=top_k
    )
    chunks_recuperados = resultados['documents'][0]
    distancias = resultados['distances'][0]
    return chunks_recuperados, distancias

pregunta = "¿Cuánto tiempo tengo para devolver un producto?"
chunks_recuperados, distancias = recuperar_contexto(pregunta, top_k=2)

print(f"❓ Pregunta: {pregunta}")
print(f"\n🔍 Chunks recuperados (top-2):")
print("=" * 60)
for i, (chunk, dist) in enumerate(zip(chunks_recuperados, distancias)):
    print(f"\n📎 Chunk {i+1} (distancia coseno: {dist:.4f}):")
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)
print("=" * 60)
print("\n💡 Distancia más baja = más similar a la pregunta")

❓ Pregunta: ¿Cuánto tiempo tengo para devolver un producto?

🔍 Chunks recuperados (top-2):

📎 Chunk 1 (distancia coseno: 0.8117):
POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos 
    desde la fecha de compra, siempre que presenten el ticket ori...

📎 Chunk 2 (distancia coseno: 1.0514):
GARANTÍA DE PRODUCTOS — TechStore Argentina
    Todos los productos tienen garantía legal de 6 meses por defectos de fabricación. 
    Los televisores y notebooks tienen garantía extendida de 12 meses...

💡 Distancia más baja = más similar a la pregunta


In [11]:
# 💬 ETAPA 2B — PROMPT AUMENTADO Y GENERACIÓN
# Armamos el prompt con el contexto recuperado y le preguntamos al LLM

def rag_completo(pregunta, top_k=2, temperature=0.3, verbose=True):

    # Recuperación
    chunks_recuperados, distancias = recuperar_contexto(pregunta, top_k)
    contexto = "\n\n".join(chunks_recuperados)

    # Prompt aumentado
    prompt_sistema = """Sos un asistente de atención al cliente de TechStore Argentina.
Respondé ÚNICAMENTE basándote en la información del contexto provisto.
Si la información no está en el contexto, decí que no tenés esa información disponible.
Respondé en español, de forma clara y concisa."""

    prompt_usuario = f"""CONTEXTO:
{contexto}

PREGUNTA DEL CLIENTE:
{pregunta}"""

    if verbose:
        print("📋 PROMPT COMPLETO QUE RECIBE EL LLM:")
        print("=" * 60)
        print(f"[SISTEMA]: {prompt_sistema}")
        print(f"\n[USUARIO]:\n{prompt_usuario}")
        print("=" * 60)

    # Generación
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": prompt_sistema},
            {"role": "user", "content": prompt_usuario}
        ],
        temperature=temperature,
        max_tokens=300
    )

    respuesta = response.choices[0].message.content
    tokens_totales = response.usage.total_tokens

    if verbose:
        print(f"\n🤖 RESPUESTA DEL SISTEMA RAG:")
        print("-" * 60)
        print(respuesta)
        print("-" * 60)
        print(f"📊 Tokens usados: {tokens_totales} | Temperature: {temperature}")

    return respuesta

# Ejecutamos el pipeline completo
rag_completo("¿Cuánto tiempo tengo para devolver un producto?")

📋 PROMPT COMPLETO QUE RECIBE EL LLM:
[SISTEMA]: Sos un asistente de atención al cliente de TechStore Argentina.
Respondé ÚNICAMENTE basándote en la información del contexto provisto.
Si la información no está en el contexto, decí que no tenés esa información disponible.
Respondé en español, de forma clara y concisa.

[USUARIO]:
CONTEXTO:
POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos 
    desde la fecha de compra, siempre que presenten el ticket original y el producto 
    esté en su embalaje original sin señales de uso.

GARANTÍA DE PRODUCTOS — TechStore Argentina
    Todos los productos tienen garantía legal de 6 meses por defectos de fabricación. 
    Los televisores y notebooks tienen garantía extendida de 12 meses. 
    La garantía no cubre daños por mal uso o accidentes.

PREGUNTA DEL CLIENTE:
¿Cuánto tiempo tengo para devolver un producto?

🤖 RESPUESTA DEL SISTEMA RAG:
------------------------

'Tienes 30 días corridos desde la fecha de compra para devolver un producto, siempre que presentes el ticket original y el producto esté en su embalaje original sin señales de uso.'

In [12]:
# ⚡ COMPARACIÓN DIRECTA: Con RAG vs Sin RAG

pregunta = "¿Me hacen envíos gratis al interior del país?"

print("🔴 SIN RAG (LLM solo):")
print("-" * 60)
sin_rag = preguntar_sin_contexto(pregunta)
print(sin_rag)

print("\n🟢 CON RAG:")
print("-" * 60)
con_rag = rag_completo(pregunta, verbose=False)
print(con_rag)

print("\n" + "=" * 60)
print("💡 La diferencia: el RAG responde con información real del corpus.")
print("   El LLM solo inventa algo plausible pero potencialmente incorrecto.")

🔴 SIN RAG (LLM solo):
------------------------------------------------------------
La disponibilidad de envíos gratis al interior del país depende de la tienda o servicio específico que estés considerando. Muchas tiendas en línea ofrecen promociones de envío gratuito si el pedido supera un cierto monto, mientras que otras pueden tener promociones especiales. Te recomendaría verificar directamente en el sitio web de la tienda o contactar su servicio al cliente para obtener información precisa sobre sus políticas de envío.

🟢 CON RAG:
------------------------------------------------------------
El envío es gratuito al interior del país solo para compras superiores a $50.000.

💡 La diferencia: el RAG responde con información real del corpus.
   El LLM solo inventa algo plausible pero potencialmente incorrecto.


---
# 📌 Parte 4 — Más allá de RAG: el paisaje actual

RAG no es la solución definitiva. Es un punto en un espectro de enfoques que el campo sigue desarrollando.

## Las alternativas y evoluciones principales

### 1. 📄 Context Stuffing
Si la ventana de contexto del LLM es suficientemente grande, ¿para qué recuperar? Simplemente metés todo el corpus en el prompt.

- **Cuándo funciona:** corpus pequeño y acotado (un contrato, un manual corto)
- **Limitación clave:** costo, latencia, y el fenómeno *"lost in the middle"* — el modelo recuerda mejor lo que está al principio y al final del contexto

### 2. 🎯 Fine-tuning
En vez de dar contexto en el prompt, el conocimiento se "hornea" directamente en los pesos del modelo durante el entrenamiento.

- **Cuándo funciona:** conocimiento estático que no cambia frecuentemente
- **Limitación clave:** el conocimiento queda congelado. Si cambia la info, hay que reentrenar
- **Hoy se usa:** en combinación con RAG, no como reemplazo

### 3. 🕸️ GraphRAG
En vez de chunks independientes, se construye un **grafo de conocimiento** con entidades y relaciones. La recuperación navega relaciones, no solo similitud vectorial.

- **Ventaja:** mucho mejor para preguntas complejas que requieren razonamiento en múltiples pasos
- **Ejemplo:** *"¿Qué productos vendidos por el proveedor X tienen problemas de garantía relacionados con el componente Y?"*

### 4. 🤖 Agentic RAG
El LLM no sigue un pipeline fijo. Decide activamente qué buscar, cuándo buscar, si necesita más información, si reformular la query.

- **El enfoque con más tracción hoy** para aplicaciones complejas
- **Frameworks:** LangGraph, LlamaIndex Workflows

### 5. ⚡ Caching Semántico
Si alguien preguntó algo similar antes, se devuelve la respuesta cacheada sin correr todo el pipeline.

- **No es un reemplazo** sino una optimización de producción
- **Reduce costos y latencia** dramáticamente en sistemas con muchos usuarios

---

## 🗺️ El mapa completo

```
  Complejidad del sistema
        ▲
        │
  Alta  │  Fine-tuning ──→ GraphRAG ──→ Agentic RAG
        │
  Media │              RAG clásico
        │
  Baja  │  Context Stuffing
        │
        └────────────────────────────────────────────▶
                                              Dinamismo del conocimiento
                                         (estático → actualización constante)
```

> **RAG clásico es el punto de entrada.** Entenderlo bien es el prerequisito para cualquiera de las evoluciones.

---
# ✅ Resumen del Bloque 1

| Concepto | Lo que aprendimos |
|---|---|
| **Alucinación** | Los LLMs generan texto plausible aunque sea incorrecto |
| **Loop autoregresivo** | El modelo genera token por token, alimentándose de su propio output |
| **Softmax + Temperature** | Convierte logits en probabilidades; temperature controla el azar |
| **Pipeline RAG** | Indexación offline + recuperación y generación en tiempo real |
| **Decisiones de diseño** | Chunking, embeddings, vector store, prompt y LLM afectan la calidad |
| **Más allá de RAG** | Context stuffing, fine-tuning, GraphRAG, Agentic RAG, caching semántico |

---

## 🚀 ¿Qué sigue?

En los próximos notebooks vamos a explorar en profundidad cada decisión de diseño:

- **Colab 1:** Chunking — ¿cómo afecta el tamaño y el overlap a la calidad del retrieval?
- **Colab 2:** Embeddings — ¿qué diferencia hay entre TF-IDF y embeddings semánticos?
- **Colab 3:** Prompt Engineering — ¿cómo cambia la respuesta según cómo le pedimos al LLM?
- **Colab 4:** Modelos y Temperature — ¿qué modelo elegir y cómo afecta la temperature?